# D04 — CloudFormation with Python

* Use S3FullAccess, GlueConsolefullAccess, should not be used in production*

This notebook creates an AWS Glue service role from a small CloudFormation YAML template. The role trusts AWS Glue and attaches `AWSGlueConsoleFullAccess`, `AmazonS3FullAccess`, and a scoped `iam:PassRole` policy.

> These managed policies are intentionally broad. Use narrower S3 and Glue permissions for production workloads. Creating a named IAM role requires the deploying identity to have CloudFormation and IAM permissions.

## AWS CLI reference

The notebook performs the deployment through Python. If the template has already been saved as `templates/glue_role.yaml`, the equivalent CLI commands are:

```bash
# Validate without creating resources.
aws cloudformation validate-template \
  --template-body file://templates/glue_role.yaml \
  --profile training \
  --region us-east-1

# Create or update the stack.
aws cloudformation deploy \
  --stack-name dataeng-glue-role \
  --template-file templates/glue_role.yaml \
  --parameter-overrides GlueRoleName=AWSGlueServiceRole-DataEngineeringLab \
  --capabilities CAPABILITY_NAMED_IAM \
  --no-fail-on-empty-changeset \
  --profile training \
  --region us-east-1
```

`CAPABILITY_NAMED_IAM` explicitly acknowledges that the stack creates a custom-named IAM role.

## 1. Imports and deployment settings

This follows the `training` profile and `us-east-1` convention used by the other AWS setup notebooks. Change these values when required. Run Jupyter from the `AWS_Setup` directory so the relative template path resolves there.

In [1]:
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

AWS_PROFILE = "training"
AWS_REGION = "us-east-1"
STACK_NAME = "dataeng-glue-role"
GLUE_ROLE_NAME = "AWSGlueServiceRole-DataEngineeringLab"
TEMPLATE_PATH = Path("templates/glue_role.yaml")


## 2. Define the CloudFormation YAML

The template is deliberately visible as a Python triple-quoted string. The trust policy lets the Glue service assume the role. `iam:PassRole` is limited to this role and only when it is passed to Glue.

In [2]:
template_yaml = """AWSTemplateFormatVersion: '2010-09-09'
Description: Creates an AWS Glue service role for the data engineering labs.

Parameters:
  GlueRoleName:
    Type: String
    Default: AWSGlueServiceRole-DataEngineeringLab
    Description: Name of the IAM role assumed by AWS Glue.
    AllowedPattern: '[A-Za-z0-9+=,.@_-]+'
    MaxLength: 64

Resources:
  GlueServiceRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: !Ref GlueRoleName
      Description: Service role used by AWS Glue jobs, crawlers, and the Glue console.
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service:
                - glue.amazonaws.com
            Action:
              - sts:AssumeRole
      ManagedPolicyArns:
        - !Sub arn:${AWS::Partition}:iam::aws:policy/AWSGlueConsoleFullAccess
        - !Sub arn:${AWS::Partition}:iam::aws:policy/AmazonS3FullAccess
      Policies:
        - PolicyName: PassThisRoleToGlue
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Sid: PassRoleToGlueOnly
                Effect: Allow
                Action:
                  - iam:PassRole
                Resource: !Sub arn:${AWS::Partition}:iam::${AWS::AccountId}:role/${GlueRoleName}
                Condition:
                  StringEquals:
                    iam:PassedToService: glue.amazonaws.com
      Tags:
        - Key: Purpose
          Value: data-engineering-lab

Outputs:
  GlueRoleName:
    Description: Name of the created Glue service role.
    Value: !Ref GlueServiceRole
  GlueRoleArn:
    Description: ARN of the created Glue service role.
    Value: !GetAtt GlueServiceRole.Arn
"""

print(template_yaml)


AWSTemplateFormatVersion: '2010-09-09'
Description: Creates an AWS Glue service role for the data engineering labs.

Parameters:
  GlueRoleName:
    Type: String
    Default: AWSGlueServiceRole-DataEngineeringLab
    Description: Name of the IAM role assumed by AWS Glue.
    AllowedPattern: '[A-Za-z0-9+=,.@_-]+'
    MaxLength: 64

Resources:
  GlueServiceRole:
    Type: AWS::IAM::Role
    Properties:
      RoleName: !Ref GlueRoleName
      Description: Service role used by AWS Glue jobs, crawlers, and the Glue console.
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service:
                - glue.amazonaws.com
            Action:
              - sts:AssumeRole
      ManagedPolicyArns:
        - !Sub arn:${AWS::Partition}:iam::aws:policy/AWSGlueConsoleFullAccess
        - !Sub arn:${AWS::Partition}:iam::aws:policy/AmazonS3FullAccess
      Policies:
        - PolicyName: PassThisRoleToGlue
  

## 3. Save the template beside the notebook

The repository already contains the same template at `AWS_Setup/templates/glue_role.yaml`. This cell makes the notebook self-contained and rewrites that file from the string above.

In [3]:
TEMPLATE_PATH.parent.mkdir(parents=True, exist_ok=True)
TEMPLATE_PATH.write_text(template_yaml, encoding="utf-8")
print(f"Saved {TEMPLATE_PATH.resolve()}")


Saved /mnt/c/Course/DataEng/AWS_Setup/templates/glue_role.yaml


## 4. Authenticate and verify the target account

Always inspect the identity and account before creating IAM resources. No secret values are printed.

In [4]:
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
identity = session.client("sts").get_caller_identity()

print(f"Account: {identity['Account']}")
print(f"Principal: {identity['Arn']}")
print(f"Region: {session.region_name}")


Account: 253862056991
Principal: arn:aws:iam::253862056991:user/cloud_user
Region: us-east-1


## 5. Validate the template

CloudFormation validation checks template structure and reports required capabilities. It does not create the role.

In [5]:
cloudformation = session.client("cloudformation")
validation = cloudformation.validate_template(TemplateBody=template_yaml)

print(validation.get("Description"))
print("Capabilities:", validation.get("Capabilities", []))
print("Parameters:", [item["ParameterKey"] for item in validation.get("Parameters", [])])


Creates an AWS Glue service role for the data engineering labs.
Capabilities: ['CAPABILITY_NAMED_IAM']
Parameters: ['GlueRoleName']


## 6. Create or update the stack

This cell changes AWS resources. It creates the stack when absent and updates it when present. A deployment with no template or parameter changes is treated as a successful no-op.

In [6]:
stack_arguments = {
    "StackName": STACK_NAME,
    "TemplateBody": template_yaml,
    "Parameters": [
        {"ParameterKey": "GlueRoleName", "ParameterValue": GLUE_ROLE_NAME}
    ],
    "Capabilities": ["CAPABILITY_NAMED_IAM"],
    "Tags": [{"Key": "Purpose", "Value": "data-engineering-lab"}],
}

try:
    cloudformation.describe_stacks(StackName=STACK_NAME)
    stack_exists = True
except ClientError as error:
    error_code = error.response.get("Error", {}).get("Code")
    if error_code == "ValidationError":
        stack_exists = False
    else:
        raise

if not stack_exists:
    response = cloudformation.create_stack(
        **stack_arguments,
        OnFailure="ROLLBACK",
    )
    print("Creating:", response["StackId"])
    cloudformation.get_waiter("stack_create_complete").wait(StackName=STACK_NAME)
    print("Stack creation completed")
else:
    try:
        response = cloudformation.update_stack(**stack_arguments)
        print("Updating:", response["StackId"])
        cloudformation.get_waiter("stack_update_complete").wait(StackName=STACK_NAME)
        print("Stack update completed")
    except ClientError as error:
        message = error.response.get("Error", {}).get("Message", "")
        if "No updates are to be performed" in message:
            print("Stack is already up to date")
        else:
            raise


Creating: arn:aws:cloudformation:us-east-1:253862056991:stack/dataeng-glue-role/de4e97a0-a6b5-11f1-ad5e-12bf9b0f9551
Stack creation completed


## 7. Read stack status and outputs

The output ARN is the role value to select when configuring a Glue job or crawler.

In [7]:
stack = cloudformation.describe_stacks(StackName=STACK_NAME)["Stacks"][0]
outputs = {item["OutputKey"]: item["OutputValue"] for item in stack.get("Outputs", [])}

print("Stack status:", stack["StackStatus"])
print("Glue role name:", outputs.get("GlueRoleName"))
print("Glue role ARN:", outputs.get("GlueRoleArn"))


Stack status: CREATE_COMPLETE
Glue role name: AWSGlueServiceRole-DataEngineeringLab
Glue role ARN: arn:aws:iam::253862056991:role/AWSGlueServiceRole-DataEngineeringLab


## 8. Troubleshoot a failed stack

Read the newest failed events first. Common causes are missing IAM permissions, an existing role with the same name, or an organization policy that blocks broad managed policies.

In [9]:
events = cloudformation.describe_stack_events(StackName=STACK_NAME)["StackEvents"]
for event in events[:15]:
    reason = event.get("ResourceStatusReason", "")
    print(event["Timestamp"], event["LogicalResourceId"], event["ResourceStatus"], reason)


2026-09-02 10:06:10.682000+00:00 dataeng-glue-role CREATE_COMPLETE 
2026-09-02 10:06:09.656000+00:00 GlueServiceRole CREATE_COMPLETE 
2026-09-02 10:05:51.935000+00:00 GlueServiceRole CREATE_IN_PROGRESS Resource creation Initiated
2026-09-02 10:05:51.037000+00:00 GlueServiceRole CREATE_IN_PROGRESS 
2026-09-02 10:05:47.462000+00:00 dataeng-glue-role CREATE_IN_PROGRESS User Initiated


## 9. Optional cleanup

Deleting the stack deletes the role created by it. First detach the role from any Glue jobs or crawlers that still use it. The cleanup call is shown but deliberately commented out.

```python
# cloudformation.delete_stack(StackName=STACK_NAME)
# cloudformation.get_waiter("stack_delete_complete").wait(StackName=STACK_NAME)
# print("Stack deleted")
```